# RCC R large-data notebook example

Use a notebook for inspection and graphics. Move full-scale computation into the provided Slurm R script when it becomes expensive.


In [ ]:
library(data.table)
set.seed(42)
dt <- data.table(
  patient_group = sample(c('A','B','C'), 100000, replace=TRUE),
  measurement = rnorm(100000),
  batch = sample(1:8, 100000, replace=TRUE)
)
head(dt)


In [ ]:
dt[, .(n=.N, mean=mean(measurement), sd=sd(measurement)), by=patient_group]


In [ ]:
library(ggplot2)
ggplot(dt[sample(.N, 5000)], aes(patient_group, measurement)) + geom_boxplot() + theme_minimal()


## RiboSnake-inspired scientific visuals

The next cells demonstrate a Bray-Curtis PCoA and a ranked waterfall plot using deterministic synthetic data. They illustrate figure construction, not biological findings.


In [ ]:
sample_count <- 36
feature_count <- 18
sample_group <- rep(c('Reference', 'Treatment A', 'Treatment B'), each=12)
abundance <- matrix(rgamma(sample_count * feature_count, shape=1.6, scale=35), nrow=sample_count)
abundance[sample_group == 'Treatment A', 1:5] <- abundance[sample_group == 'Treatment A', 1:5] * 2.2
abundance[sample_group == 'Treatment B', 6:10] <- abundance[sample_group == 'Treatment B', 6:10] * 2.4
relative_abundance <- abundance / rowSums(abundance)

bray_curtis <- function(x) {
  n <- nrow(x)
  distances <- matrix(0, n, n)
  for (i in seq_len(n - 1)) {
    for (j in (i + 1):n) {
      distances[i, j] <- distances[j, i] <- sum(abs(x[i, ] - x[j, ])) / sum(x[i, ] + x[j, ])
    }
  }
  as.dist(distances)
}
ordination <- cmdscale(bray_curtis(relative_abundance), k=2, eig=TRUE, add=TRUE)
positive_eigenvalues <- ordination$eig[ordination$eig > 0]
explained <- 100 * ordination$eig[1:2] / sum(positive_eigenvalues)
pcoa <- data.table(ordination$points, sample_group)
setnames(pcoa, c('PCoA1', 'PCoA2', 'sample_group'))


In [ ]:
ggplot(pcoa, aes(PCoA1, PCoA2, color=sample_group, shape=sample_group)) +
  geom_hline(yintercept=0, color='grey85') +
  geom_vline(xintercept=0, color='grey85') +
  geom_point(size=3, alpha=0.85) +
  scale_color_manual(values=c('Reference'='#4477AA', 'Treatment A'='#EE6677', 'Treatment B'='#228833')) +
  labs(
    title='Synthetic community profiles separate by treatment',
    x=sprintf('PCoA 1 (%.1f%% of positive eigenvalue sum)', explained[1]),
    y=sprintf('PCoA 2 (%.1f%% of positive eigenvalue sum)', explained[2]),
    color='Sample group', shape='Sample group'
  ) +
  theme_minimal(base_size=12) + theme(legend.position='bottom')


A PCoA is descriptive and needs statistical testing plus scientific interpretation. The waterfall plot below orders every synthetic sample and marks an example threshold chosen before interpreting the values.


In [ ]:
waterfall <- data.table(
  sample_id=sprintf('S%02d', 1:24),
  relative_change=pmax(-58, pmin(45, rnorm(24, mean=-8, sd=22)))
)[order(relative_change)]
waterfall[, response := ifelse(relative_change <= -20, 'Meets example threshold', 'Does not meet threshold')]
waterfall[, sample_id := factor(sample_id, levels=sample_id)]

ggplot(waterfall, aes(sample_id, relative_change, fill=response)) +
  geom_col(width=0.82) +
  geom_hline(yintercept=0, linewidth=0.4) +
  geom_hline(yintercept=-20, color='#4477AA', linetype='dashed') +
  scale_fill_manual(values=c('Meets example threshold'='#228833', 'Does not meet threshold'='#CC6677')) +
  labs(
    title='Ranked synthetic response by sample',
    x='Sample, ordered by relative change', y='Relative change from baseline (%)', fill=NULL
  ) +
  theme_minimal(base_size=12) +
  theme(axis.text.x=element_text(angle=60, hjust=1), legend.position='bottom')
